In [3]:
import sys
import json
import random
import os
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

import napari
from scipy.interpolate import griddata
from pathlib import Path

from plot_utils_era5 import *

project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root / 'src'))
sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

sys.path.append(f"{project_root}src/")
# Import utilities
from utilities import KernelNN, KernelNN_subgraph, GraphLMDBReader, setup_device, get_output_affinities
from plot_utilities import plot_results_napari, plot_results_matplotlib

sys.path.append(f"{project_root}/ERA5/scripts/")
from utilities_era5 import ERA5_Graph_Generator as Data_Generator
from utilities_era5 import ERA5_Graph_Generator_perSampleNorm as Data_Generator_old
data_path = project_root / 'ERA5' / 'data'

# Device setup
device = setup_device()
print(f'Using device: {device}')

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import subprocess


def load_model(id:str, model_type:str='random_crop'):
    model_path = project_root / 'ERA5' / 'trained_models' / model_type / f'{id}.pth'
    config_path = project_root / 'ERA5' / 'trained_models' / model_type / 'logs' / 'configs' / f'{id}.json' 
    
    with open(config_path, 'r') as f:
        config = json.load(f)
    
    model = KernelNN(
        width=config['width'],
        ker_width=config['ker_width'],
        depth=config['depth'],
        ker_in=config['edge_attr'],
        in_width=config['node_features'],
        out_width=config['out_width']
    ).to(device)
    
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    return model, config

using Device: mps
Using device: mps


In [4]:
model, config = load_model(id='test1_model5_median_tkernel_2026-04-22_t21-22-37' , model_type='regular')

In [5]:
lat_min = config["lat_min"]
lat_max = config["lat_max"]
lon_min = config["lon_min"]
lon_max = config["lon_max"]
#N = config["N"]
k = config["k"]
perplexity = config["perplexity"]
sampling_method = config["sampling_method"]
train_date_start_end = config["train_date_start_end"]
test_date_start_end = config["test_date_start_end"]
train_samples_per_date = config["train_samples_per_date"]
test_samples_per_date = config["test_samples_per_date"]
feature_mode = config["feature_mode"]
vars = config["vars"]


In [6]:
dates = pd.date_range("2019-01-01", "2019-01-05").strftime("%Y-%m-%d").tolist()

#Generator = Data_Generator_old(data_path)
Generator = Data_Generator(data_path)

bbox = lon_min, lat_min, lon_max, lat_max

N=1000
print('generate data')

graphs=[]
locations_list=[]
X_list=[]

for date in dates:
    Generator.load_data(bbox=bbox, vars=vars, filename=f'{date}.nc')
    print(f'total points available: {Generator.da.shape[1]*Generator.da.shape[2]}')
    graph = Generator.sample_graph(N=N, k=k, perplexity=perplexity, sampling_method=sampling_method, gen_feature_mode=feature_mode, sigma_method='ones')
    sample = graph.to(device)
    graphs.append(sample)
    locations_list.append(pd.DataFrame({"lat": sample.meta.gps[:,0], "lon": sample.meta.gps[:,1]}))
    X_list.append(sample.normed_values.detach().cpu().numpy())

generate data
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761
total points available: 32761


In [7]:
pred_nofe_list = []
pred_pca_list = []
pred_tsne_list = []
pred_umap_list = []

for sample in graphs:
    # NOFE
    print('NOFE ...', end='\r')
    pred_nofe = model(sample).detach().cpu().numpy()
    pred_nofe_list.append(pred_nofe)
    
    X = sample.normed_values.detach().cpu().numpy()
    
    # PCA
    print('PCA ...', end='\r')
    pca = PCA(n_components=3)
    pred_pca = pca.fit_transform(X)
    pred_pca_list.append(pred_pca)

    # t-SNE
    print('t-SNE ...', end='\r')
    pred_tsne = TSNE(
        n_components=3,
        perplexity=30,
        learning_rate="auto",
        random_state=42
    ).fit_transform(X)
    pred_tsne_list.append(pred_tsne)

    # UMAP
    print('UMAP ...', end='\r')
    umap_id = 'umap_buffer'
    dir = f'{project_root}/ERA5/umap'
    np.save(f'{dir}/{umap_id}.npy', X)
    subprocess.run(f"cd {dir} && conda run -n umap python execute_umap.py {umap_id}", shell=True)
    pred_umap = np.load(f'{dir}/res_{umap_id}.npy')
    pred_umap_list.append(pred_umap)

    os.remove(f'{dir}/{umap_id}.npy')
    os.remove(f'{dir}/res_{umap_id}.npy')


importing libraries
reading in data
applying umap
saving file
"
file saved as res_umap_buffer.npy"



/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/__init__.py:36: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/Users/laueb4818/miniforge3/envs/umap/lib/python3.10/site-packages/umap/umap_.py:1943: UserWarning: n_jobs value -1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(f"n_jobs value {self.n_jobs} overridden to 1 by setting random_state. Use no seed for parallelism.")




CondaError: KeyboardInterrupt



KeyboardInterrupt: 

In [8]:
i = 1
H = graphs[i].normed_values.detach().cpu().numpy()
L = pred_nofe_list[i]

In [1]:
os.getcwd()

'/Volumes/T7-storage/NOFE/ERA5/notebooks'

In [10]:
def pairwise_distances(X):
    # X: (N, d)
    norms = np.sum(X**2, axis=1, keepdims=True)  # (N, 1)
    
    dist2 = norms + norms.T - 2 * X @ X.T        # (N, N)
    
    # numerical stability: avoid tiny negatives
    dist2 = np.maximum(dist2, 0.0)
    
    return np.sqrt(dist2)

In [ ]:
def distance_preservation_score(H, L):
    DH = pairwise_distances(H)
    DL = pairwise_distances(L)
    
    # Take upper triangle (avoid duplicates + diagonal)
    i = np.triu_indices_from(DH, k=1)
    dH = DH[i]
    dL = DL[i]
    
    # Correlation (1 = perfect preservation)
    corr = np.corrcoef(dH, dL)[0, 1]
    
    # stress 
    stress = np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))
    
    return corr, stress

0.7318118869578327

In [ ]:
# alternative: stress
stress = np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

In [ ]:

def edge_distances(X, edge_indices):
    # X: (N, d), edge_indices: (E, 2)
    diff = X[edge_indices[:, 0]] - X[edge_indices[:, 1]]
    return np.linalg.norm(diff, axis=1)

def edge_distance_preservation(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    # Correlation
    corr = np.corrcoef(dH, dL)[0, 1]
    return corr

def edge_stress(H, L, edge_indices):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    return np.sqrt(np.sum((dH - dL)**2) / np.sum(dH**2))

def weighted_edge_stress(H, L, edge_indices, weights):
    dH = edge_distances(H, edge_indices)
    dL = edge_distances(L, edge_indices)
    
    return np.sqrt(np.sum(weights * (dH - dL)**2) / np.sum(weights * dH**2))

In [22]:
pairwise_distances(H).flatten()

array([0.       , 0.9481411, 1.3830543, ..., 3.7229054, 1.1992848,
       0.       ], dtype=float32)

In [27]:
i = 1
H = graphs[i].normed_values.detach().cpu().numpy()
Lnofe = pred_nofe_list[i]
Lpca = pred_pca_list[i]
Ltsne = pred_tsne_list[i]
Lumap = pred_umap_list[i]

print(f'nofe: {np.corrcoef(pairwise_distances(H).flatten(), pairwise_distances(Lnofe).flatten())[0,1]}')
print(f'pca: {np.corrcoef(pairwise_distances(H).flatten(), pairwise_distances(Lpca).flatten())[0,1]}')
print(f'tsne: {np.corrcoef(pairwise_distances(H).flatten(), pairwise_distances(Ltsne).flatten())[0,1]}')
print(f'umap: {np.corrcoef(pairwise_distances(H).flatten(), pairwise_distances(Lumap).flatten())[0,1]}')

nofe: 0.7973486879798041
pca: 0.8792023032627031
tsne: 0.5386682086435883
umap: 0.46418500417194025


In [6]:
def pairwise_dist_local():
    pass

def pairwise_dist_global():
    pass

In [19]:
# A
dist_H = pairwise_distances(H)
dist_L = pairwise_distances(L)
distance_preservation_score(dist_H, dist_L)

(0.7318118869578327, 0.3298196)